In [ ]:

import sys
import pandas as pd

# Check the active Python interpreter and pandas version
print("Python executable:", sys.executable)
print("Pandas version:", pd.__version__)


In [ ]:

import json
from pathlib import Path
import pandas as pd

# Find the raw JSON file
file_path = Path("data/raw/booking_jobs.json")

if not file_path.exists():
    file_path = Path("../data/raw/booking_jobs.json")

# Load job postings
with file_path.open(encoding="utf-8") as file:
    data = json.load(file)

print("Total jobs:", data["totalCount"])
print("Loaded jobs:", len(data["jobs"]))


In [ ]:

# Extract job attributes from the nested JSON
jobs = [job["data"] for job in data["jobs"]]

# Convert the list of dictionaries into a DataFrame
df = pd.DataFrame(jobs)

# Display the first five rows
df.head()


In [ ]:

# Select relevant job attributes
columns = [
    "req_id",
    "title",
    "city",
    "department",
    "posted_date",
    "salary_min_value",
    "salary_max_value",
]

df[columns].head(10)


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:

# Calculate missing values for each column
missing_values = pd.DataFrame({
    "non_null_count": df.notna().sum(),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
})

# Sort columns by the percentage of missing values
missing_values = missing_values.sort_values(
    by="missing_pct",
    ascending=False
)

# Display the 15 columns with the most missing values
missing_values.head(15)


In [ ]:
missing_values

In [ ]:

# Select fields relevant to job market analysis
selected_columns = [
    "req_id",
    "title",
    "city",
    "department",
    "employment_type",
    "posted_date",
    "salary_min_value",
    "salary_max_value",
]

# Display the actual values
df[selected_columns].head(10)


In [ ]:

# Count empty or whitespace-only strings
empty_strings = df[selected_columns].apply(
    lambda column: column.map(
        lambda value: isinstance(value, str) and not value.strip()
    ).sum()
)

empty_strings[empty_strings > 0]


In [ ]:

# Create a copy of the raw DataFrame
df_clean = df.copy()

# Replace empty strings with missing values
df_clean["department"] = df_clean["department"].replace(
    r"^\s*$", pd.NA, regex=True
)

# Convert publication dates to datetime
df_clean["posted_date"] = pd.to_datetime(
    df_clean["posted_date"],
    errors="coerce",
    utc=True
)

# Treat zero salary values as missing for exploratory analysis
for column in ["salary_min_value", "salary_max_value"]:
    df_clean[column] = df_clean[column].mask(
        df_clean[column] == 0
    )

# Inspect the cleaned data
df_clean[
    [
        "title",
        "city",
        "department",
        "posted_date",
        "salary_min_value",
        "salary_max_value",
    ]
].head(10)


In [ ]:
df_clean[
    [
        "department",
        "posted_date",
        "salary_min_value",
        "salary_max_value",
    ]
].info()

In [ ]:

import requests

url = "https://jobs.booking.com/api/jobs"

# Request the second page of job postings
response = requests.get(
    url,
    params={"page": 2, "limit": 10},
    timeout=20,
)

response.raise_for_status()

page_2 = response.json()

# Display the number of returned jobs
print("Jobs on page 2:", len(page_2["jobs"]))
print("Total jobs:", page_2["totalCount"])

# Compare job IDs between the first and second pages
page_1_ids = {
    job["data"]["req_id"]
    for job in data["jobs"]
}

page_2_ids = {
    job["data"]["req_id"]
    for job in page_2["jobs"]
}

print("Overlapping IDs:", len(page_1_ids & page_2_ids))


In [ ]:

import json
from pathlib import Path
import pandas as pd

# Find the complete dataset
file_path = Path("data/raw/booking_jobs_all.json")

if not file_path.exists():
    file_path = Path("../data/raw/booking_jobs_all.json")

# Load all collected job postings
with file_path.open(encoding="utf-8") as file:
    raw_data = json.load(file)

# Convert nested JSON into a DataFrame
df_all = pd.DataFrame(
    job["data"] for job in raw_data["jobs"]
)

print("Total jobs:", len(df_all))
print("Unique job IDs:", df_all["req_id"].nunique())


In [ ]:

# Count job postings by city
city_distribution = (
    df_all["city"]
    .value_counts(dropna=False)
    .reset_index()
)

city_distribution.columns = ["city", "job_count"]

city_distribution.head(20)


In [ ]:

# Filter job postings located in Amsterdam
amsterdam_mask = (
    df_all["city"]
    .fillna("")
    .str.strip()
    .str.casefold()
    .eq("amsterdam")
)

df_amsterdam = df_all.loc[amsterdam_mask].copy()

print("Total Booking.com jobs:", len(df_all))
print("Jobs in Amsterdam:", len(df_amsterdam))

# Display available positions
df_amsterdam[
    ["req_id", "title", "city", "posted_date"]
].sort_values("posted_date", ascending=False)


In [ ]:

# Display all Amsterdam job titles
with pd.option_context("display.max_rows", None):
    display(
        df_amsterdam[["req_id", "title"]]
        .sort_values("title")
        .reset_index(drop=True)
    )


In [ ]:

# Define keywords for potential analytics-related roles
keywords = [
    "analyst",
    "analytics",
    "data",
    "insights",
    "business intelligence",
    "data scientist",
    "experimentation",
]

# Build a regular expression from the keywords
pattern = "|".join(keywords)

# Find job titles containing at least one keyword
df_candidates = df_amsterdam[
    df_amsterdam["title"].str.contains(
        pattern,
        case=False,
        na=False
    )
].copy()

print("Potentially relevant jobs:", len(df_candidates))

display(
    df_candidates[["req_id", "title", "posted_date"]]
    .sort_values("title")
)


In [ ]:

# Display complete titles without truncation
for _, job in df_candidates.sort_values("title").iterrows():
    print(f'{job["req_id"]}: {job["title"]}')


In [ ]:

# Select a specific job by its ID
job_id = "30143"

selected_job = df_candidates.loc[
    df_candidates["req_id"].astype(str) == job_id
].iloc[0]

print("Title:", selected_job["title"])
print("\nDescription preview:")
print(selected_job["description"][:1500])


In [ ]:

from bs4 import BeautifulSoup

# Extract the original HTML description
html_description = selected_job["description"]

# Parse HTML and extract readable text
soup = BeautifulSoup(html_description, "html.parser")

clean_description = soup.get_text(
    separator="\n",
    strip=True
)

# Display the cleaned description
print(clean_description[:4000])


In [ ]:

import re

# Define skills and their matching patterns
skill_patterns = {
    "SQL": r"\bSQL\b",
    "Python": r"\bPython\b",
    "Financial Modeling": r"\bfinancial models?\b",
    "Business Cases": r"\bbusiness cases?\b",
    "Market Analysis": r"\bmarket trends?\b",
    "Competitor Analysis": r"\bcompetitor activity\b",
    "Stakeholder Management": r"\bstakeholders?\b",
}

# Extract skills mentioned in the job description
found_skills = []

for skill, pattern in skill_patterns.items():
    if re.search(pattern, clean_description, flags=re.IGNORECASE):
        found_skills.append(skill)

print("Detected skills:")
for skill in found_skills:
    print("-", skill)


In [ ]:

# Find all SQL mentions and display their context
for match in re.finditer(
    skill_patterns["SQL"],
    clean_description,
    flags=re.IGNORECASE
):
    start = max(0, match.start() - 150)
    end = min(len(clean_description), match.end() + 150)

    print(clean_description[start:end])
    print("-" * 50)


In [ ]:

def extract_skills(description: str) -> list[str]:
    # Find all skills matching the defined patterns
    found_skills = []

    for skill, pattern in skill_patterns.items():
        if re.search(pattern, description, flags=re.IGNORECASE):
            found_skills.append(skill)

    return found_skills


In [ ]:
extract_skills(clean_description)

In [ ]:

from bs4 import BeautifulSoup

# Create a separate DataFrame for skill analysis
df_skills = df_amsterdam.copy()

# Convert HTML descriptions into plain text
df_skills["description_text"] = df_skills["description"].apply(
    lambda html: BeautifulSoup(
        html if isinstance(html, str) else "",
        "html.parser"
    ).get_text(" ", strip=True)
)

# Extract skills from each job description
df_skills["skills"] = df_skills["description_text"].apply(
    extract_skills
)

# Preview extracted skills
df_skills[["title", "skills"]].head(10)


In [ ]:

# Define technical and analytical skill patterns
skill_patterns = {
    "SQL": r"\bSQL\b",
    "Python": r"\bPython\b",
    "Tableau": r"\bTableau\b",
    "Power BI": r"\bPower\s*BI\b",
    "Financial Modeling": r"\bfinancial model(?:s|ing)?\b",
    "Business Cases": r"\bbusiness cases?\b",
    "Market Analysis": r"\bmarket (?:trends?|analysis|research)\b",
    "Competitor Analysis": r"\bcompetitor (?:activity|analysis|research)\b",
}


In [ ]:

# Recalculate extracted skills using updated patterns
df_skills["skills"] = df_skills["description_text"].apply(
    extract_skills
)

# Inspect the Commercial Strategy Analyst position
display(
    df_skills.loc[
        df_skills["req_id"].astype(str) == "30143",
        ["title", "skills"]
    ]
)


In [ ]:
print(
    df_skills.loc[
        df_skills["req_id"].astype(str) == "30143",
        "skills"
    ].iloc[0]
)

In [ ]:

# Create a dataset for manual role classification
df_roles = df_amsterdam[
    ["req_id", "title", "description"]
].copy()

# Mark jobs selected by the initial keyword filter
df_roles["keyword_match"] = df_roles["req_id"].isin(
    df_candidates["req_id"]
)

# Display all jobs without truncating titles
with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None
):
    display(
        df_roles[
            ["req_id", "title", "keyword_match"]
        ].sort_values(
            ["keyword_match", "title"],
            ascending=[False, True]
        ).reset_index(drop=True)
    )


In [ ]:

# Define manually reviewed job categories
role_labels = {
    "30143": "strategy_analytics",
    "29992": "financial_analytics",
    "30320": "analytics_engineering",
    "30066": "analytics_engineering",
    "30161": "data_engineering",
    "30097": "data_science",
    "28789": "machine_learning",
    "30379": "machine_learning",
    "29369": "machine_learning",
    "29529": "machine_learning",
    "29400": "machine_learning",
    "29985": "quantitative_research",
    "30383": "needs_review",
}

# Assign categories using job IDs
df_roles["role_category"] = (
    df_roles["req_id"]
    .astype(str)
    .map(role_labels)
    .fillna("other")
)

# Count jobs by category
df_roles["role_category"].value_counts()


In [ ]:
df_roles

In [ ]:

from bs4 import BeautifulSoup

# Select the job that requires manual review
job_id = "30383"

job = df_roles.loc[
    df_roles["req_id"].astype(str) == job_id
].iloc[0]

# Convert the HTML description into plain text
description = BeautifulSoup(
    job["description"],
    "html.parser"
).get_text(separator="\n", strip=True)

print("Job title:", job["title"])
print("\nJob description:\n")
print(description[:5000])


In [ ]:

# Update the manually reviewed job category
role_labels["30383"] = "data_governance"

# Reapply manual labels to the DataFrame
df_roles["role_category"] = (
    df_roles["req_id"]
    .astype(str)
    .map(role_labels)
    .fillna("unreviewed")
)

# Verify the updated classification
df_roles.loc[
    df_roles["req_id"].astype(str) == "30383",
    ["req_id", "title", "role_category"]
]


In [ ]:
df_roles